In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
class BasicBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()
    # if stride = 1 and padding = 1 (out_channels, H, W) else (out_channels, H//2, W//2)
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False) # (out_channels, H, W) or (out_channels, H//2, W//2)
    self.bn1 = nn.BatchNorm2d(out_channels) # (out_channels, H, W) or (out_channels, H//2, W//2)
    self.relu = nn.ReLU(inplace=True) # (out_channels, H, W) or (out_channels, H//2, W//2)

    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False) # (out_channels, H, W) or (out_channels, H//2, W//2)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.shortcut = nn.Sequential()

    # 1 x 1 conv
    if stride!=1 or in_channels != out_channels:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels)
      )

  def forward(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)
    out = self.conv2(out)
    out = self.bn2(out)
    out = out + self.shortcut(x)
    out = self.relu(out)
    return out

In [10]:
class ResNet18(nn.Module):
  def __init__(self, num_classes=10):
    super().__init__()
    self.in_channels = 64

    # first convolution
    self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) # (64, H, W)
    self.bn1 = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)

    self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1) # 64 is output channels, 2 blocks, stride=1 (no downsampling)
    self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2) # downsampling, inc channels size
    self.layer2 = self._make_layer(BasicBlock, 256, 2, stride=2)
    self.layer2 = self._make_layer(BasicBlock, 512, 2, stride=2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) # (512,)
    self.fc = nn.Linear(512, num_classes)

  def _make_layer(self, block, out_channels, num_blocks, stride):
    strides = [stride] + [1]*(num_blocks-1) # [stride, 1, 1, ...]
    layers = []
    for stride in strides:
      layers.append(block(self.in_channels, out_channels, stride))
      self.in_channels = out_channels
    return nn.Sequential(*layers)

  def forward(self, x):
    out = self.conv1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.layer1(out)
    out = self.layer2(out)
    out = self.layer3(out)
    out = self.layer4(out)

    out = self.avgpool(out)
    out = out.view(out.size(0), -1)
    out = self.fc(out)
    return out

model = ResNet18().to(device)
print(model)

ResNet18(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=